In [1]:

# ── Load feeds (drop last open bar) ─────────────────────────────────────────
import pandas as pd
import numpy as np
import plotly.graph_objects as go
from plotly.subplots import make_subplots
from pathlib import Path

def find_project_root(marker=".git"):
    p = Path.cwd()
    while p != p.parent:
        if (p / marker).exists():
            return p
        p = p.parent
    raise FileNotFoundError("Project root not found")

symbol  = "BTCUSDT"
raw_dir = find_project_root() / "data" / "mlData" / "raw"

df_1h  = pd.read_json(raw_dir / f"{symbol}-1h-vX.jsonl",  lines=True, convert_dates=False).iloc[:-1]
df_15m = pd.read_json(raw_dir / f"{symbol}-15m-vX.jsonl", lines=True, convert_dates=False).iloc[:-1]

df_1h["dt"]  = pd.to_datetime(df_1h["timestamp"],  unit="ms")
df_15m["dt"] = pd.to_datetime(df_15m["timestamp"], unit="ms")

print(f"1H  : {len(df_1h):,} bars  {df_1h['dt'].iloc[0].date()} → {df_1h['dt'].iloc[-1].date()}")
print(f"15m : {len(df_15m):,} bars  {df_15m['dt'].iloc[0].date()} → {df_15m['dt'].iloc[-1].date()}")


1H  : 45,954 bars  2020-12-11 → 2026-03-10
15m : 182,817 bars  2020-12-21 → 2026-03-10


In [2]:
# ── Compute indicators ────────────────────────────────────────────────────────
def add_ema(df, periods, col="close"):
    for p in periods:
        df[f"ema_{p}"] = df[col].ewm(span=p, adjust=False).mean()
    return df

def add_atr_ratio(df, short=10, long=50):
    tr = pd.concat([
        df["high"] - df["low"],
        (df["high"] - df["close"].shift()).abs(),
        (df["low"]  - df["close"].shift()).abs(),
    ], axis=1).max(axis=1)
    df["atr_short"] = tr.ewm(span=short, adjust=False).mean()
    df["atr_long"]  = tr.ewm(span=long,  adjust=False).mean()
    df["atr_ratio"] = df["atr_short"] / df["atr_long"]
    return df

def add_rsi(df, period=30, col="close"):
    delta    = df[col].diff()
    gain     = delta.clip(lower=0)
    loss     = -delta.clip(upper=0)
    avg_gain = gain.ewm(alpha=1 / period, adjust=False).mean()
    avg_loss = loss.ewm(alpha=1 / period, adjust=False).mean()
    rs = avg_gain / avg_loss
    df["rsi"] = 100 - 100 / (1 + rs)
    return df

def add_taker_buy_ratio(df):
    df["taker_buy_ratio"] = df["taker-buy-basevol"] / df["vol"].replace(0, np.nan)
    return df

# 1H: EMA 50/200 for group, RSI30 (not used in sub-state but useful for plots)
df_1h = add_ema(df_1h, [50, 200])
df_1h = add_rsi(df_1h, period=30)

# 1H group from EMA crossover
df_1h["group"] = np.where(df_1h["ema_50"] > df_1h["ema_200"], "BULLISH", "BEARISH")

# Warm-up: first 200 1H bars — EMA initialisation error non-negligible (safe fetch floor)
WARMUP_1H_BARS = 200
df_1h["warmup"] = False
df_1h.iloc[:WARMUP_1H_BARS, df_1h.columns.get_loc("warmup")] = True

# 15m: ATR 10/50, RSI30, taker_buy_ratio
df_15m = add_ema(df_15m, [20, 60])
df_15m = add_atr_ratio(df_15m, short=10, long=50)
df_15m = add_rsi(df_15m, period=30)
df_15m = add_taker_buy_ratio(df_15m)

print("1H group distribution (raw, no confirmation buffer):")
print(df_1h["group"].value_counts().to_string())
warmup_end = df_1h.iloc[WARMUP_1H_BARS - 1]["dt"].date()
print(f"\nWarm-up window : first {WARMUP_1H_BARS} 1H bars  "
      f"({df_1h.iloc[0]['dt'].date()} → {warmup_end})")

1H group distribution (raw, no confirmation buffer):
group
BULLISH    23935
BEARISH    22019

Warm-up window : first 200 1H bars  (2020-12-11 → 2020-12-19)


In [3]:
# ── Merge 1H group into 15m (backward asof join on timestamp) ────────────────
df_1h_grp = df_1h[["timestamp", "group", "warmup"]].sort_values("timestamp")
df_15m    = df_15m.sort_values("timestamp")

df_15m = pd.merge_asof(
    df_15m,
    df_1h_grp.rename(columns={"timestamp": "ts_1h"}),
    left_on="timestamp",
    right_on="ts_1h",
    direction="backward",
)

# Any 15m bar that precedes the first 1H bar gets group=NaN → treat as warm-up
df_15m["warmup"] = df_15m["warmup"].fillna(True)
df_15m["group"]  = df_15m["group"].fillna("UNKNOWN")

print(f"15m bars after merge : {len(df_15m):,}")
print(f"Warm-up 15m bars     : {int(df_15m['warmup'].sum()):,}")
print("\n15m group distribution (post warm-up):")
print(df_15m[~df_15m["warmup"]]["group"].value_counts().to_string())

15m bars after merge : 182,817
Warm-up 15m bars     : 0

15m group distribution (post warm-up):
group
BULLISH    94755
BEARISH    88062


In [4]:
# ── Regime classifier (row-by-row with hysteresis) ───────────────────────────
DIVERGE_LOOKBACK = 20   # Issue 2: lookback for RSI > 60 peak (15m bars)

def classify_regimes(df):
    rsi    = df["rsi"].values
    atr    = df["atr_ratio"].values
    tbr    = df["taker_buy_ratio"].values
    group  = df["group"].values
    warmup = df["warmup"].values.astype(bool)

    n      = len(df)
    labels = np.empty(n, dtype=object)
    prev   = "UNKNOWN"

    for i in range(n):
        if warmup[i]:
            labels[i] = "UNKNOWN_WARMUP"
            prev = "UNKNOWN_WARMUP"
            continue

        g = group[i]
        r = rsi[i]
        a = atr[i]
        t = tbr[i]

        if np.isnan(r) or np.isnan(a):
            labels[i] = prev
            continue

        regime = None

        if g == "BULLISH":
            if 48 <= r <= 55 and a < 1.0:
                regime = "ACCUMULATION"
            elif r > 55 and a < 1.3:
                regime = "BULLISH"
            else:
                # RECOVERY: RSI just crossed 48↑, ATR 1.0–1.5, ATR declining
                rsi_crossed  = (i > 0) and (rsi[i - 1] < 48) and (r >= 48)
                atr_in_range = 1.0 <= a <= 1.5
                atr_declining = (i >= 4) and (a <= atr[i - 4])
                if rsi_crossed and atr_in_range and atr_declining:
                    regime = "RECOVERY"

        elif g == "BEARISH":
            if r < 40 and a > 1.5:
                regime = "CORRECTION"
            elif 40 <= r <= 48 and 1.0 <= a <= 1.3:
                regime = "BEARISH"
            else:
                # DISTRIBUTION: RSI diverging ↓ from > 60, ATR expanding (1.0–1.3), tbr < 0.45
                if 1.0 <= a <= 1.3 and (not np.isnan(t)) and t < 0.45:
                    start     = max(0, i - DIVERGE_LOOKBACK)
                    rsi_peak  = np.nanmax(rsi[start : i + 1])
                    if rsi_peak > 60 and r < rsi_peak:
                        regime = "DISTRIBUTION"

        # Gap or no match → hysteresis: hold previous
        if regime is None:
            regime = prev

        labels[i] = regime
        prev = regime

    return pd.Series(labels, index=df.index, name="regime")

df_15m = df_15m.reset_index(drop=True)
df_15m["regime"] = classify_regimes(df_15m)

print("Regime counts (all bars):")
print(df_15m["regime"].value_counts().to_string())

Regime counts (all bars):
regime
BEARISH         60252
ACCUMULATION    57698
BULLISH         35633
DISTRIBUTION    16876
CORRECTION       8322
RECOVERY         4014
UNKNOWN            22


In [5]:
# ── Regime distribution analysis ─────────────────────────────────────────────
import plotly.express as px

REGIME_ORDER  = ["ACCUMULATION", "BULLISH", "RECOVERY",
                 "DISTRIBUTION", "BEARISH",  "CORRECTION", "UNKNOWN"]
REGIME_COLORS = {
    "ACCUMULATION": "#66BB6A",
    "BULLISH":      "#2E7D32",
    "RECOVERY":     "#A5D6A7",
    "DISTRIBUTION": "#EF9A9A",
    "BEARISH":      "#C62828",
    "CORRECTION":   "#FF7043",
    "UNKNOWN":      "#B0BEC5",
}

warmup_mask = df_15m["regime"] == "UNKNOWN_WARMUP"
post_mask   = ~warmup_mask

warmup_n = int(warmup_mask.sum())
post_n   = int(post_mask.sum())
total_n  = len(df_15m)

print(f"Total 15m bars : {total_n:,}")
if warmup_n > 0:
    warmup_start = df_15m[warmup_mask].iloc[0]["dt"].date()
    warmup_end   = df_15m[warmup_mask].iloc[-1]["dt"].date()
    print(f"Warm-up bars   : {warmup_n:,}  ({warmup_start} → {warmup_end})")
else:
    # 15m dataset starts after the 1H EMA warm-up window ends — no overlap
    wu_end_1h = df_1h.iloc[WARMUP_1H_BARS - 1]["dt"].date()
    print(f"Warm-up bars   : 0  (1H warm-up ended {wu_end_1h}; 15m data starts {df_15m.iloc[0]['dt'].date()})")
print(f"Post warm-up   : {post_n:,}\n")

# Count post-warm-up regimes (UNKNOWN = genuine gap held by hysteresis)
counts = df_15m[post_mask]["regime"].value_counts()

print(f"{'Regime':<16} {'Count':>8}  {'%':>6}")
print("-" * 34)
for regime in REGIME_ORDER:
    c   = counts.get(regime, 0)
    pct = 100 * c / post_n
    print(f"{regime:<16} {c:>8,}  {pct:>5.1f}%")
print("-" * 34)
print(f"{'Total':<16} {post_n:>8,}  100.0%")

# Bar chart
plot_df = (
    counts
    .reindex(REGIME_ORDER, fill_value=0)
    .reset_index()
)
plot_df.columns = ["regime", "count"]
plot_df["pct"]  = 100 * plot_df["count"] / post_n

wu_note = f"warm-up: 0 bars (ended before 15m data)" if warmup_n == 0 else f"warm-up excluded: {warmup_n:,} bars"
fig = px.bar(
    plot_df,
    x="regime", y="pct",
    color="regime",
    color_discrete_map=REGIME_COLORS,
    text=plot_df["pct"].map(lambda x: f"{x:.1f}%"),
    title=f"Regime distribution — {post_n:,} bars  |  {wu_note}",
    labels={"pct": "% of bars", "regime": ""},
    category_orders={"regime": REGIME_ORDER},
)
fig.update_traces(textposition="outside")
fig.update_layout(showlegend=False, height=450, yaxis=dict(range=[0, 60]))
fig.show()

Total 15m bars : 182,817
Warm-up bars   : 0  (1H warm-up ended 2020-12-19; 15m data starts 2020-12-21)
Post warm-up   : 182,817

Regime              Count       %
----------------------------------
ACCUMULATION       57,698   31.6%
BULLISH            35,633   19.5%
RECOVERY            4,014    2.2%
DISTRIBUTION       16,876    9.2%
BEARISH            60,252   33.0%
CORRECTION          8,322    4.6%
UNKNOWN                22    0.0%
----------------------------------
Total             182,817  100.0%


In [6]:
n_train = int(len(df_15m) * 0.80)
n_val   = int(len(df_15m) * 0.10)

splits = {
    "train": df_15m.iloc[:n_train],
    "val":   df_15m.iloc[n_train : n_train + n_val],
    "test":  df_15m.iloc[n_train + n_val:],
}
for name, split in splits.items():
    print(f"\n{name}  ({split.iloc[0]['dt'].date()} → {split.iloc[-1]['dt'].date()})")
    c = split["regime"].value_counts()
    for r in REGIME_ORDER:
        print(f"  {r:<16} {c.get(r,0):>7,}  {100*c.get(r,0)/len(split):>5.1f}%")



train  (2020-12-21 → 2025-02-23)
  ACCUMULATION      47,262   32.3%
  BULLISH           28,910   19.8%
  RECOVERY           3,023    2.1%
  DISTRIBUTION      12,410    8.5%
  BEARISH           48,074   32.9%
  CORRECTION         6,552    4.5%
  UNKNOWN               22    0.0%

val  (2025-02-23 → 2025-09-01)
  ACCUMULATION       6,034   33.0%
  BULLISH            3,944   21.6%
  RECOVERY             648    3.5%
  DISTRIBUTION       2,020   11.0%
  BEARISH            4,944   27.0%
  CORRECTION           691    3.8%
  UNKNOWN                0    0.0%

test  (2025-09-01 → 2026-03-10)
  ACCUMULATION       4,402   24.1%
  BULLISH            2,779   15.2%
  RECOVERY             343    1.9%
  DISTRIBUTION       2,446   13.4%
  BEARISH            7,234   39.6%
  CORRECTION         1,079    5.9%
  UNKNOWN                0    0.0%
